# Wav2Vec2.0 Encoder Validation Report
### Research Meeting — PD Voice Analysis

**Architecture flow:**
`Audio → Temporal Segmentation (early/middle/late) → Wav2Vec2.0 Encoder → F_early, F_middle, F_late → [NEXT: Gating Network]`

**Goal:** Validate that the encoder produces *meaningful* embeddings before building the Gating Network on top.

In [1]:
import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.spatial.distance import cosine as cosine_dist
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE      = os.path.abspath(os.getcwd())
META_PATH = os.path.join(BASE, "metadata.csv")
RESULTS   = os.path.join(BASE, "results", "encoder_validation")
REPORT    = os.path.join(BASE, "results", "encoder_validation_report.txt")
os.makedirs(RESULTS, exist_ok=True)

CLASSES  = ["PD", "HC"]
SEGMENTS = ["early", "middle", "late"]
report_lines = []

def rprint(*args):
    msg = " ".join(str(a) for a in args)
    print(msg)
    report_lines.append(msg)

def save_fig(fig, name):
    path = os.path.join(RESULTS, name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"  Figure saved -> {path}")
    plt.close(fig)

# ── Load embeddings ───────────────────────────────────────────────────────────
print("Loading metadata...")
meta = pd.read_csv(META_PATH)

embeddings, labels, segments, stems = [], [], [], []
missing = 0
for _, row in meta.iterrows():
    p = row["embedding_path"]
    if not os.path.isfile(p):
        missing += 1; continue
    v = np.load(p)
    if v.ndim == 2:
        v = v.mean(axis=0)
    if not np.isfinite(v).all():
        missing += 1; continue
    embeddings.append(v)
    labels.append(row["class"])
    segments.append(row["segment"])
    stems.append(str(row.get("original_stem", "")))

X      = np.stack(embeddings)
y_cls  = np.array(labels)
y_seg  = np.array(segments)
y_stem = np.array(stems)

print(f"Loaded {len(X)} embeddings  dim={X.shape[1]}  missing/corrupt={missing}")
print(f"  PD={( y_cls=='PD').sum()}  HC={(y_cls=='HC').sum()}")

Loading metadata...


Loaded 3741 embeddings  dim=768  missing/corrupt=0
  PD=1014  HC=2727


---
## Section 0 — Data Sanity Check

> Before any analysis, we verify our data is loaded correctly. **Garbage in = garbage out.**
> We check embedding shapes, norms, variance, and NaN/Inf values for all 6 groups.

In [2]:
rprint("\n" + "="*60)
rprint("SECTION 0: DATA SANITY CHECK")
rprint("="*60)

header = f"{'Group':<14} | {'Count':>6} | {'Emb Dim':>7} | {'Mean Norm':>10} | {'Std Norm':>9}"
sep    = "-" * len(header)
rprint(header); rprint(sep)

any_warn = False
for cls in CLASSES:
    for seg in SEGMENTS:
        mask  = (y_cls == cls) & (y_seg == seg)
        grp   = X[mask]
        n     = len(grp)
        dim   = str(grp.shape[1]) if n > 0 else "—"
        norms = np.linalg.norm(grp, axis=1) if n > 0 else np.array([0.0])
        mn    = f"{norms.mean():.4f}"
        sn    = f"{norms.std():.4f}"
        rprint(f"{cls+'/'+seg:<14} | {n:>6} | {dim:>7} | {mn:>10} | {sn:>9}")

        if n < 5:
            rprint(f"  WARNING: Too few samples in {cls}/{seg}")
            any_warn = True
        if n > 0 and norms.mean() < 1e-3:
            rprint(f"  WARNING: Dead embeddings in {cls}/{seg} (norm ~ 0)")
            any_warn = True
        if n > 1 and grp.std(axis=0).mean() < 1e-6:
            rprint(f"  WARNING: No variance in {cls}/{seg}")
            any_warn = True
        if n > 0 and not np.isfinite(grp).all():
            rprint(f"  WARNING: Corrupt embeddings in {cls}/{seg} (NaN/Inf)")
            any_warn = True

rprint(sep)
rprint("" if any_warn else "  OK: No red flags detected. Data looks healthy.")


SECTION 0: DATA SANITY CHECK
Group          |  Count | Emb Dim |  Mean Norm |  Std Norm
----------------------------------------------------------
PD/early       |    338 |     768 |     8.2232 |    0.4762
PD/middle      |    338 |     768 |     8.4115 |    0.4166
PD/late        |    338 |     768 |     8.4380 |    0.4393
HC/early       |    909 |     768 |     8.3091 |    0.4631
HC/middle      |    909 |     768 |     8.4582 |    0.4809
HC/late        |    909 |     768 |     8.4773 |    0.4967
----------------------------------------------------------
  OK: No red flags detected. Data looks healthy.


---
## Section 1 — PD vs HC Separation
### Can the encoder tell them apart at all?

> This is the most fundamental test. If Wav2Vec2.0 embeddings cannot separate PD from HC even slightly,
> there is no point building a Gating Network on top. We need at least some separation here.

In [3]:
rprint("\nRunning Section 1A: t-SNE PD vs HC...")

X_sc   = StandardScaler().fit_transform(X)
coords = TSNE(n_components=2, perplexity=30, random_state=42,
              init="pca", max_iter=1000).fit_transform(X_sc)

fig, ax = plt.subplots(figsize=(8, 6))
C_MAP   = {"PD": "#E53935", "HC": "#1E88E5"}
M_MAP   = {"PD": "o",       "HC": "^"}
for cls in CLASSES:
    m = y_cls == cls
    ax.scatter(coords[m,0], coords[m,1], label=f"{cls} (n={m.sum()})",
               alpha=0.55, s=14, color=C_MAP[cls], marker=M_MAP[cls], linewidths=0)
ax.set_title("PD vs HC — Wav2Vec2.0 Encoder Output", fontsize=13, fontweight="bold")
ax.set_xlabel("t-SNE Component 1"); ax.set_ylabel("t-SNE Component 2")
ax.legend(title="Class", markerscale=2, framealpha=0.8)
plt.tight_layout()
save_fig(fig, "1A_tsne_pd_vs_hc.png")

# Auto-detect cluster separation via inter/intra-class centroid ratio
pd_c = coords[y_cls=="PD"].mean(0)
hc_c = coords[y_cls=="HC"].mean(0)
inter = np.linalg.norm(pd_c - hc_c)
intra = ((np.linalg.norm(coords[y_cls=="PD"] - pd_c, axis=1).mean() +
          np.linalg.norm(coords[y_cls=="HC"] - hc_c, axis=1).mean()) / 2)
ratio = inter / (intra + 1e-9)

if ratio > 1.5:
    sep_desc  = "well-separated"
    enc_desc  = "is clearly"
elif ratio > 0.7:
    sep_desc  = "partially overlapping"
    enc_desc  = "partially is"
else:
    sep_desc  = "fully mixed"
    enc_desc  = "is not"

rprint(f"  Cluster separation ratio: {ratio:.3f}")
rprint(f"  Clusters are [{sep_desc}]")
rprint(f"  This means the encoder [{enc_desc}] capturing disease-related voice features")


Running Section 1A: t-SNE PD vs HC...


  Figure saved -> /home/bs00956/Desktop/Personal/PD/Pipeline-Implementation/Wav2Vec2/results/encoder_validation/1A_tsne_pd_vs_hc.png
  Cluster separation ratio: 0.109
  Clusters are [fully mixed]
  This means the encoder [is not] capturing disease-related voice features


In [4]:
rprint("\nRunning Section 1B: Linear Separability Score...")

y_bin = (y_cls == "PD").astype(int)
pipe  = Pipeline([("sc", StandardScaler()),
                  ("clf", LogisticRegression(max_iter=1000, solver="lbfgs",
                                             class_weight="balanced"))])
skf   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
accs  = cross_val_score(pipe, X, y_bin, cv=skf, scoring="accuracy")
acc   = accs.mean() * 100
std   = accs.std()  * 100

rprint(f"\n  Accuracy: {acc:.1f}% +/- {std:.1f}%")
rprint("")
if acc >= 80:
    verdict = ("STRONG: Encoder clearly separates PD from HC. "
               "Safe to proceed to Gating Network.")
elif acc >= 65:
    verdict = ("MODERATE: Encoder has some signal. "
               "Gating Network may improve this further.")
elif acc >= 50:
    verdict = ("WEAK: Encoder barely separates. "
               "Consider fine-tuning Wav2Vec2.0 before adding Gating.")
else:
    verdict = ("POOR: Encoder is not capturing PD features. "
               "Do NOT proceed to Gating. Diagnose first.")

rprint(f"  Verdict: {verdict}")
overall_acc = acc


Running Section 1B: Linear Separability Score...



  Accuracy: 68.7% +/- 0.6%

  Verdict: MODERATE: Encoder has some signal. Gating Network may improve this further.


In [5]:
rprint("\nRunning Section 1C: Confusion Matrix...")

y_bin  = (y_cls == "PD").astype(int)
y_pred = cross_val_predict(pipe, X, y_bin, cv=skf)
cm     = confusion_matrix(y_bin, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
disp    = ConfusionMatrixDisplay(cm, display_labels=["HC", "PD"])
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Confusion Matrix — PD vs HC (5-fold CV)", fontweight="bold")
plt.tight_layout()
save_fig(fig, "1C_confusion_matrix.png")

tn, fp, fn, tp = cm.ravel()
rprint(f"  Model confuses {fn} PD patients as HC (missed detections)")
rprint(f"  Model confuses {fp} HC as PD (false alarms)")
rprint("  Clinical note: In medical context, missing PD patients is more dangerous than false alarms")


Running Section 1C: Confusion Matrix...


  Figure saved -> /home/bs00956/Desktop/Personal/PD/Pipeline-Implementation/Wav2Vec2/results/encoder_validation/1C_confusion_matrix.png
  Model confuses 414 PD patients as HC (missed detections)
  Model confuses 758 HC as PD (false alarms)
  Clinical note: In medical context, missing PD patients is more dangerous than false alarms


---
## Section 2 — Segment Behaviour Analysis
### Does the encoder capture the early → middle → late progression?

> Your architecture splits audio into early/middle/late specifically because **PD voice changes over time**.
> If the encoder captures this progression, we will see F_early, F_middle, F_late moving in embedding space.
> If not, all three will look identical — meaning temporal segmentation adds no value.

In [6]:
rprint("\nRunning Section 2A: Segment t-SNE per class...")

SEG_C = {"early": "#4CAF50", "middle": "#FF9800", "late": "#9C27B0"}
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, cls in zip(axes, CLASSES):
    mask    = y_cls == cls
    c_coord = coords[mask]
    c_seg   = y_seg[mask]
    for seg in SEGMENTS:
        sm = c_seg == seg
        ax.scatter(c_coord[sm,0], c_coord[sm,1], label=f"{seg} (n={sm.sum()})",
                   alpha=0.6, s=14, color=SEG_C[seg], linewidths=0)
    ax.set_title(f"{cls} — coloured by Segment", fontweight="bold")
    ax.set_xlabel("t-SNE C1"); ax.set_ylabel("t-SNE C2")
    ax.legend(title="Segment", markerscale=2, framealpha=0.8)

plt.suptitle("Segment Distribution within each Class", fontsize=13, fontweight="bold")
plt.tight_layout()
save_fig(fig, "2A_segment_tsne_per_class.png")

for cls in CLASSES:
    mask    = y_cls == cls
    c_coord = coords[mask]
    c_seg   = y_seg[mask]
    centroids = {seg: c_coord[c_seg==seg].mean(0) for seg in SEGMENTS}
    spread    = np.mean([np.linalg.norm(c_coord[c_seg==s] - centroids[s], axis=1).mean()
                         for s in SEGMENTS])
    inter_d   = np.linalg.norm(centroids["early"] - centroids["late"])
    if inter_d > spread * 0.5:
        rprint(f"  {cls}: Segments are [separated] -> progression [captured]")
    else:
        rprint(f"  {cls}: Segments are [mixed] -> progression [not captured]")


Running Section 2A: Segment t-SNE per class...


  Figure saved -> /home/bs00956/Desktop/Personal/PD/Pipeline-Implementation/Wav2Vec2/results/encoder_validation/2A_segment_tsne_per_class.png
  PD: Segments are [separated] -> progression [captured]
  HC: Segments are [separated] -> progression [captured]


In [7]:
rprint("\nRunning Section 2B: Progression Trajectory...")

# Centroids in original 768-d space
centroids_high = {}
for cls in CLASSES:
    for seg in SEGMENTS:
        mask = (y_cls == cls) & (y_seg == seg)
        centroids_high[f"{cls}_{seg}"] = X[mask].mean(0)

# Centroid projections in t-SNE space (for arrow diagram)
centroids_2d = {}
for cls in CLASSES:
    for seg in SEGMENTS:
        mask = (y_cls == cls) & (y_seg == seg)
        centroids_2d[f"{cls}_{seg}"] = coords[mask].mean(0)

# Cosine distance: early vs late
PD_prog = cosine_dist(centroids_high["PD_early"], centroids_high["PD_late"])
HC_prog = cosine_dist(centroids_high["HC_early"], centroids_high["HC_late"])
rprint(f"  PD progression (early->late cosine dist) : {PD_prog:.4f}")
rprint(f"  HC progression (early->late cosine dist) : {HC_prog:.4f}")

# Arrow diagram in t-SNE space
fig, ax = plt.subplots(figsize=(8, 6))
for cls in CLASSES:
    m = y_cls == cls
    ax.scatter(coords[m,0], coords[m,1], alpha=0.2, s=10,
               color=C_MAP[cls], linewidths=0)

for cls, col in [("PD","#E53935"), ("HC","#1E88E5")]:
    pts = [centroids_2d[f"{cls}_{s}"] for s in SEGMENTS]
    for i in range(len(SEGMENTS)-1):
        dx = pts[i+1][0] - pts[i][0]
        dy = pts[i+1][1] - pts[i][1]
        ax.annotate("", xy=pts[i+1], xytext=pts[i],
                    arrowprops=dict(arrowstyle="->", color=col, lw=2))
    for j, seg in enumerate(SEGMENTS):
        ax.scatter(*pts[j], s=120, color=col, zorder=5,
                   edgecolors="white", linewidths=1.5)
        ax.annotate(f"{cls[:2]}-{seg[:2]}", pts[j], fontsize=8,
                    ha="center", va="bottom", color=col)

ax.set_title("Progression Trajectory: early -> middle -> late", fontweight="bold")
ax.set_xlabel("t-SNE C1"); ax.set_ylabel("t-SNE C2")
handles = [mpatches.Patch(color=C_MAP[c], label=c) for c in CLASSES]
ax.legend(handles=handles, title="Class")
plt.tight_layout()
save_fig(fig, "2B_progression_trajectory.png")

if PD_prog > HC_prog:
    rprint("\n  GOOD: PD voice changes more than HC across segments.")
    rprint("  Temporal segmentation is capturing disease progression.")
    rprint("  Gating Network will have meaningful differences to work with.")
    pd_prog_good = True
else:
    rprint("\n  PROBLEM: PD and HC show similar progression patterns.")
    rprint("  Possible causes:")
    rprint("    1. Wav2Vec2.0 not sensitive to PD-specific changes")
    rprint("    2. Segmentation boundaries may need adjustment")
    rprint("    3. Consider fine-tuning encoder")
    pd_prog_good = False


Running Section 2B: Progression Trajectory...
  PD progression (early->late cosine dist) : 0.0893
  HC progression (early->late cosine dist) : 0.0826


  Figure saved -> /home/bs00956/Desktop/Personal/PD/Pipeline-Implementation/Wav2Vec2/results/encoder_validation/2B_progression_trajectory.png

  GOOD: PD voice changes more than HC across segments.
  Temporal segmentation is capturing disease progression.
  Gating Network will have meaningful differences to work with.


In [8]:
rprint("\nRunning Section 2C: Per-segment PD vs HC Accuracy...")

y_bin = (y_cls == "PD").astype(int)
seg_accs = {}
rprint(f"\n  {'Segment':<10} | {'Accuracy':>10} | {'Best for detection?':>20}")
rprint("  " + "-"*47)

for seg in SEGMENTS:
    mask = y_seg == seg
    Xs   = X[mask]
    ys   = y_bin[mask]
    p    = Pipeline([("sc", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=1000,
                                                class_weight="balanced"))])
    sk   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    a    = cross_val_score(p, Xs, ys, cv=sk, scoring="accuracy").mean() * 100
    seg_accs[seg] = a
    best = "YES - most discriminative" if a == max(seg_accs.values()) else ""
    rprint(f"  {seg:<10} | {a:>9.1f}% | {best:>20}")

best_seg = max(seg_accs, key=seg_accs.get)
rprint(f"\n  The [{best_seg}] segment carries the most discriminative information.")
rprint(f"  Gating Network should learn to weight this segment higher.")


Running Section 2C: Per-segment PD vs HC Accuracy...

  Segment    |   Accuracy |  Best for detection?
  -----------------------------------------------


  early      |      68.8% | YES - most discriminative


  middle     |      66.8% |                     


  late       |      63.9% |                     

  The [early] segment carries the most discriminative information.
  Gating Network should learn to weight this segment higher.


---
## Section 3 — Encoder Quality Deep Dive
### What problems exist and how to fix them?

> Before the meeting, we want to know not just IF there are problems but **WHAT** problems and **HOW** to fix them.

In [9]:
rprint("\nRunning Section 3A: Within-class Consistency...")

sim_data = {}
for cls in CLASSES:
    for seg in SEGMENTS:
        mask = (y_cls == cls) & (y_seg == seg)
        grp  = X[mask]
        if len(grp) < 2:
            sim_data[f"{cls}_{seg}"] = 0.0
            continue
        # sample up to 300 for speed
        idx  = np.random.RandomState(42).choice(len(grp), min(300, len(grp)), replace=False)
        sim  = cosine_similarity(grp[idx])
        np.fill_diagonal(sim, np.nan)
        sim_data[f"{cls}_{seg}"] = np.nanmean(sim)

fig, ax = plt.subplots(figsize=(8, 5))
x_pos   = np.arange(len(SEGMENTS))
width   = 0.35
for i, cls in enumerate(CLASSES):
    vals = [sim_data[f"{cls}_{s}"] for s in SEGMENTS]
    bars = ax.bar(x_pos + i*width - width/2, vals, width,
                  label=cls, color=C_MAP[cls], alpha=0.8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=8)
ax.set_xticks(x_pos); ax.set_xticklabels(SEGMENTS)
ax.set_xlabel("Segment"); ax.set_ylabel("Mean Pairwise Cosine Similarity")
ax.set_title("Within-class Consistency per Segment", fontweight="bold")
ax.legend(title="Class"); ax.set_ylim(0, 1.0)
plt.tight_layout()
save_fig(fig, "3A_within_class_consistency.png")

pd_mean_sim = np.mean([sim_data[f"PD_{s}"] for s in SEGMENTS])
hc_mean_sim = np.mean([sim_data[f"HC_{s}"] for s in SEGMENTS])
all_sims    = list(sim_data.values())

if hc_mean_sim > pd_mean_sim:
    rprint("  EXPECTED: HC voice is more consistent than PD (healthy speakers more uniform)")
else:
    rprint("  UNEXPECTED: PD embeddings more consistent than HC. May indicate encoder not capturing disease variation")
if np.mean(all_sims) > 0.95:
    rprint("  WARNING: Embeddings may be too similar (low variance). Encoder might be collapsing inputs")
elif np.mean(all_sims) < 0.3:
    rprint("  WARNING: Embeddings too different even within same class. Possible noise problem")


Running Section 3A: Within-class Consistency...


  Figure saved -> /home/bs00956/Desktop/Personal/PD/Pipeline-Implementation/Wav2Vec2/results/encoder_validation/3A_within_class_consistency.png
  EXPECTED: HC voice is more consistent than PD (healthy speakers more uniform)


In [10]:
rprint("\nRunning Section 3B: Problem Diagnosis Checklist...")

problems_found = []
rprint(f"\n  {'Check':<35} | {'Result':>6} | Details")
rprint("  " + "-"*70)

# CHECK 1: Class imbalance
pd_n  = (y_cls=="PD").sum()
hc_n  = (y_cls=="HC").sum()
ratio = max(pd_n, hc_n) / (min(pd_n, hc_n) + 1e-9)
c1    = ratio > 2.0
if c1: problems_found.append("Class imbalance")
rprint(f"  {'CHECK 1: Class imbalance?':<35} | {'YES' if c1 else 'NO':>6} | PD={pd_n} HC={hc_n} ratio={ratio:.1f}{'  -> Use class_weight=balanced' if c1 else ''}")

# CHECK 2: Embedding collapse
var = X.var(axis=0).mean()
c2  = var < 0.01
if c2: problems_found.append("Embedding collapse")
rprint(f"  {'CHECK 2: Embedding collapse?':<35} | {'YES' if c2 else 'NO':>6} | mean variance={var:.5f}{'  -> Near-constant vectors' if c2 else ''}")

# CHECK 3: Segment overlap
early_c  = X[y_seg=="early"].mean(0)
middle_c = X[y_seg=="middle"].mean(0)
late_c   = X[y_seg=="late"].mean(0)
d_em = cosine_dist(early_c, middle_c)
d_ml = cosine_dist(middle_c, late_c)
c3   = (d_em < 0.05) and (d_ml < 0.05)
if c3: problems_found.append("Segment overlap")
rprint(f"  {'CHECK 3: Segment overlap?':<35} | {'YES' if c3 else 'NO':>6} | early-mid={d_em:.4f} mid-late={d_ml:.4f}{'  -> Adjust boundaries' if c3 else ''}")

# CHECK 4: Feature saturation (PCA)
pca     = PCA(random_state=42).fit(StandardScaler().fit_transform(X))
cum_var = np.cumsum(pca.explained_variance_ratio_)
n_for90 = int(np.searchsorted(cum_var, 0.90)) + 1
c4      = n_for90 <= 3
if c4: problems_found.append("Feature saturation")
rprint(f"  {'CHECK 4: Feature saturation?':<35} | {'YES' if c4 else 'NO':>6} | 90% var in {n_for90} PCs{'  -> Try attention pooling' if c4 else ''}")

# CHECK 5: Speaker identity dominance
# compare within-speaker vs between-speaker same-class similarity
per_speaker = {}
for i, stem in enumerate(y_stem):
    if stem not in per_speaker:
        per_speaker[stem] = []
    per_speaker[stem].append(i)

within_sp, between_sp = [], []
rng = np.random.RandomState(42)
for stem, idxs in list(per_speaker.items())[:100]:
    if len(idxs) < 2: continue
    grp_vecs = X[idxs]
    s = cosine_similarity(grp_vecs)
    np.fill_diagonal(s, np.nan)
    within_sp.append(np.nanmean(s))
    # between-speaker same class
    cls0 = y_cls[idxs[0]]
    other_idx = np.where((y_cls == cls0) & ~np.isin(np.arange(len(y_cls)), idxs))[0]
    if len(other_idx) >= 2:
        sample = rng.choice(other_idx, min(10, len(other_idx)), replace=False)
        s2 = cosine_similarity(grp_vecs, X[sample])
        between_sp.append(s2.mean())

c5 = len(within_sp) > 0 and len(between_sp) > 0 and      np.mean(within_sp) - np.mean(between_sp) > 0.15
if c5: problems_found.append("Speaker identity dominance")
ws_val = f"{np.mean(within_sp):.4f}" if within_sp else "N/A"
bs_val = f"{np.mean(between_sp):.4f}" if between_sp else "N/A"
rprint(f"  {'CHECK 5: Speaker dominance?':<35} | {'YES' if c5 else 'NO':>6} | within-speaker={ws_val} between={bs_val}{'  -> Add speaker norm' if c5 else ''}")
rprint("")
rprint(f"  Problems found: {len(problems_found)} -> {', '.join(problems_found) if problems_found else 'NONE'}")


Running Section 3B: Problem Diagnosis Checklist...

  Check                               | Result | Details
  ----------------------------------------------------------------------
  CHECK 1: Class imbalance?           |    YES | PD=1014 HC=2727 ratio=2.7  -> Use class_weight=balanced
  CHECK 2: Embedding collapse?        |     NO | mean variance=0.05431
  CHECK 3: Segment overlap?           |     NO | early-mid=0.1528 mid-late=0.0157


  CHECK 4: Feature saturation?        |     NO | 90% var in 69 PCs
  CHECK 5: Speaker dominance?         |     NO | within-speaker=0.5419 between=0.4008

  Problems found: 1 -> Class imbalance


In [11]:
rprint("\nSection 3C: Recommended Fixes")

rprint(f"\n  {'Problem Found':<25} | Recommended Fix")
rprint("  " + "-"*75)
fixes = {
    "Class imbalance":       "Add class_weight='balanced' to all classifiers",
    "Embedding collapse":    "Try averaging last 4 transformer layers instead of last layer only",
    "Segment overlap":       "Adjust cut points — try 0-2s / 2-6s / 6-10s",
    "Feature saturation":    "Try attention pooling instead of mean pooling over time",
    "Speaker identity dominance": "Add per-speaker mean subtraction (speaker normalization)",
}
if not problems_found:
    rprint("  No problems found — encoder quality looks good.")
else:
    for prob in problems_found:
        rprint(f"  {prob:<25} | {fixes.get(prob,'')}")


Section 3C: Recommended Fixes

  Problem Found             | Recommended Fix
  ---------------------------------------------------------------------------
  Class imbalance           | Add class_weight='balanced' to all classifiers


---
## Section 4 — Meeting-Ready Summary Slide

> Formatted report suitable for presenting at the research meeting.

In [12]:
rprint("\nRunning Section 4: Generating Summary Report...")

total_n = len(X)
pd_n    = (y_cls=="PD").sum()
hc_n    = (y_cls=="HC").sum()

if overall_acc >= 80:
    gating_ready  = "YES"
    confidence    = "HIGH"
elif overall_acc >= 65:
    gating_ready  = "FIX FIRST (optional)"
    confidence    = "MEDIUM"
else:
    gating_ready  = "FIX FIRST"
    confidence    = "LOW"

prog_str = "YES" if pd_prog_good else "NO"
best_seg_str = max(seg_accs, key=seg_accs.get)

next_steps = []
if "Class imbalance" in problems_found:
    next_steps.append("Apply class_weight=balanced to all downstream classifiers")
if "Speaker identity dominance" in problems_found:
    next_steps.append("Add per-speaker mean subtraction before Gating Network")
if "Segment overlap" in problems_found:
    next_steps.append("Revisit segmentation boundaries (current: 0-3s/3-7s/7-10s)")
if not next_steps:
    next_steps = [
        f"Proceed to Gating Network design (encoder validated)",
        f"Weight [{best_seg_str}] segment higher in initial Gating Network",
    ]
while len(next_steps) < 2:
    next_steps.append("Monitor encoder performance on held-out test set")

prob_lines = (problems_found + ["NONE", "NONE", "NONE"])[:3]

report = f"""
╔══════════════════════════════════════════════════════╗
║      WAV2VEC2.0 ENCODER VALIDATION REPORT           ║
║      Research Meeting — PD Voice Analysis           ║
╠══════════════════════════════════════════════════════╣
║ DATASET                                             ║
║   Total samples : {total_n:<6} (PD: {pd_n:<5} HC: {hc_n:<5})       ║
║   Segments/sample: 3 (early / middle / late)        ║
║   Embedding dim  : 768                              ║
╠══════════════════════════════════════════════════════╣
║ ENCODER PERFORMANCE                                 ║
║   PD vs HC accuracy (all segments): {overall_acc:.1f}%         ║
║   Best segment for detection      : {best_seg_str:<14}  ║
║   PD progression captured        : {prog_str:<14}  ║
╠══════════════════════════════════════════════════════╣
║ PROBLEMS IDENTIFIED                                 ║
║   1. {prob_lines[0]:<48}║
║   2. {prob_lines[1]:<48}║
║   3. {prob_lines[2]:<48}║
╠══════════════════════════════════════════════════════╣
║ VERDICT                                             ║
║   Ready for Gating Network : {gating_ready:<24}║
║   Confidence level         : {confidence:<24}║
╠══════════════════════════════════════════════════════╣
║ RECOMMENDED NEXT STEPS                              ║
║   1. {next_steps[0][:48]:<48}║
║   2. {next_steps[1][:48]:<48}║
╚══════════════════════════════════════════════════════╝
"""
rprint(report)

# Save report to file
os.makedirs(os.path.dirname(REPORT), exist_ok=True)
with open(REPORT, "w") as f:
    f.write("\n".join(report_lines) + "\n" + report)
print(f"\nReport saved -> {REPORT}")
print(f"Figures saved -> {RESULTS}/")


Running Section 4: Generating Summary Report...

╔══════════════════════════════════════════════════════╗
║      WAV2VEC2.0 ENCODER VALIDATION REPORT           ║
║      Research Meeting — PD Voice Analysis           ║
╠══════════════════════════════════════════════════════╣
║ DATASET                                             ║
║   Total samples : 3741   (PD: 1014  HC: 2727 )       ║
║   Segments/sample: 3 (early / middle / late)        ║
║   Embedding dim  : 768                              ║
╠══════════════════════════════════════════════════════╣
║ ENCODER PERFORMANCE                                 ║
║   PD vs HC accuracy (all segments): 68.7%         ║
║   Best segment for detection      : early           ║
║   PD progression captured        : YES             ║
╠══════════════════════════════════════════════════════╣
║ PROBLEMS IDENTIFIED                                 ║
║   1. Class imbalance                                 ║
║   2. NONE                                        

---
## Section 5 — Extended Analysis

Three additional checks the original report did not include:

- **5A** — `class_weight=balanced` vs default: explicit PD recall comparison (note: Section 1B already used balanced — here we show the full trade-off)
- **5B** — Aggregated embeddings: concat of early+middle+late → 2304-dim vector. Does aggregation help?
- **5C** — Layer analysis: current mean-pool of **last layer** vs **avg of last 4 transformer layers**. Does using more layers help?

In [13]:
from sklearn.metrics import recall_score, precision_score, f1_score

print("Running Section 5A: Default vs class_weight=balanced comparison...")
print()

y_bin5 = (y_cls == "PD").astype(int)
skf5   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

pipe_def = Pipeline([("sc", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=1000, solver="lbfgs",
                                                random_state=42))])
pipe_bal = Pipeline([("sc", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=1000, solver="lbfgs",
                                                random_state=42,
                                                class_weight="balanced"))])

pred_def = cross_val_predict(pipe_def, X, y_bin5, cv=skf5)
pred_bal = cross_val_predict(pipe_bal, X, y_bin5, cv=skf5)

def show_report(y_true, y_pred, label):
    cm     = confusion_matrix(y_true, y_pred)
    acc    = accuracy_score(y_true, y_pred) * 100
    rec    = recall_score(y_true, y_pred)   * 100
    pre    = precision_score(y_true, y_pred, zero_division=0) * 100
    f1     = f1_score(y_true, y_pred, zero_division=0) * 100
    tn, fp, fn, tp = cm.ravel()
    print(f"  [{label}]")
    print(f"    Accuracy   : {acc:.1f}%")
    print(f"    PD Recall  : {rec:.1f}%   <- % of PD patients correctly identified")
    print(f"    Precision  : {pre:.1f}%")
    print(f"    F1 Score   : {f1:.1f}%")
    print(f"    Missed PD  : {fn} patients labelled as HC (false negatives)")
    print(f"    False alarm: {fp} HC labelled as PD (false positives)")
    return acc, rec, f1

print("COMPARISON — Default (no weight) vs class_weight='balanced'")
print("-" * 60)
acc_d, rec_d, f1_d = show_report(y_bin5, pred_def, "Default — no class_weight")
print()
acc_b, rec_b, f1_b = show_report(y_bin5, pred_bal, "class_weight='balanced'")
print()
print(f"  Delta Accuracy : {acc_d:.1f}% -> {acc_b:.1f}%  ({acc_b - acc_d:+.1f} pp)")
print(f"  Delta PD Recall: {rec_d:.1f}% -> {rec_b:.1f}%  ({rec_b - rec_d:+.1f} pp)")
print(f"  Delta F1 Score : {f1_d:.1f}% -> {f1_b:.1f}%  ({f1_b - f1_d:+.1f} pp)")
print()
if rec_b > rec_d + 2:
    print("  VERDICT: class_weight=balanced significantly improves PD recall.")
    print("  Trade-off: accuracy drops slightly — acceptable for clinical use.")
    print("  Missing PD is more dangerous than false alarms. Use balanced.")
elif rec_b > rec_d:
    print("  VERDICT: class_weight=balanced slightly improves PD recall.")
    print("  Use balanced in all downstream classifiers as a safe default.")
else:
    print("  VERDICT: balanced did not improve PD recall for this dataset split.")
    print("  Both strategies similar. Keep balanced as safe clinical default.")

Running Section 5A: Default vs class_weight=balanced comparison...



COMPARISON — Default (no weight) vs class_weight='balanced'
------------------------------------------------------------
  [Default — no class_weight]
    Accuracy   : 73.4%
    PD Recall  : 42.8%   <- % of PD patients correctly identified
    Precision  : 51.1%
    F1 Score   : 46.6%
    Missed PD  : 580 patients labelled as HC (false negatives)
    False alarm: 415 HC labelled as PD (false positives)

  [class_weight='balanced']
    Accuracy   : 68.7%
    PD Recall  : 59.2%   <- % of PD patients correctly identified
    Precision  : 44.2%
    F1 Score   : 50.6%
    Missed PD  : 414 patients labelled as HC (false negatives)
    False alarm: 758 HC labelled as PD (false positives)

  Delta Accuracy : 73.4% -> 68.7%  (-4.7 pp)
  Delta PD Recall: 42.8% -> 59.2%  (+16.4 pp)
  Delta F1 Score : 46.6% -> 50.6%  (+4.0 pp)

  VERDICT: class_weight=balanced significantly improves PD recall.
  Trade-off: accuracy drops slightly — acceptable for clinical use.
  Missing PD is more dangerous than f

In [14]:
print("Running Section 5B: Aggregated embeddings test...")
print()

META_AGG = os.path.join(BASE, "metadata_aggregated.csv")
meta_agg = pd.read_csv(META_AGG)
print(f"Aggregated metadata: {len(meta_agg)} rows  |  strategy=concat  |  dim=2304")
print(f"  PD={( meta_agg['class']=='PD').sum()}  HC={(meta_agg['class']=='HC').sum()}")
print()

X_agg, y_agg = [], []
miss_agg = 0
for _, row in meta_agg.iterrows():
    p = row["embedding_path"]
    if not os.path.isfile(p):
        miss_agg += 1
        continue
    v = np.load(p)
    if v.ndim == 2:
        v = v.mean(axis=0)
    if not np.isfinite(v).all():
        miss_agg += 1
        continue
    X_agg.append(v)
    y_agg.append(row["class"])

X_agg     = np.stack(X_agg)
y_agg_arr = np.array(y_agg)
y_agg_bin = (y_agg_arr == "PD").astype(int)
print(f"Loaded: {len(X_agg)}  missing/corrupt: {miss_agg}  dim={X_agg.shape[1]}")
print()

skf_agg   = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
pipe_agg_def = Pipeline([("sc", StandardScaler()),
                          ("clf", LogisticRegression(max_iter=2000, solver="lbfgs",
                                                     random_state=42))])
pipe_agg_bal = Pipeline([("sc", StandardScaler()),
                          ("clf", LogisticRegression(max_iter=2000, solver="lbfgs",
                                                     random_state=42,
                                                     class_weight="balanced"))])

acc_agg_def = cross_val_score(pipe_agg_def, X_agg, y_agg_bin, cv=skf_agg,
                               scoring="accuracy").mean() * 100
acc_agg_bal = cross_val_score(pipe_agg_bal, X_agg, y_agg_bin, cv=skf_agg,
                               scoring="accuracy").mean() * 100
rec_agg_bal = cross_val_score(pipe_agg_bal, X_agg, y_agg_bin, cv=skf_agg,
                               scoring="recall").mean()   * 100
f1_agg_bal  = cross_val_score(pipe_agg_bal, X_agg, y_agg_bin, cv=skf_agg,
                               scoring="f1").mean()       * 100

print("  AGGREGATED (concat 3x768=2304-dim) vs PER-SEGMENT (768-dim):")
print("  " + "-" * 58)
print(f"  {'Method':<40}  {'Accuracy':>10}")
print("  " + "-" * 58)
print(f"  {'Per-segment, default (Section 1B baseline)':<40}  {acc_d:>9.1f}%")
print(f"  {'Per-segment, balanced (Section 5A)':<40}  {acc_b:>9.1f}%")
print(f"  {'Aggregated, default':<40}  {acc_agg_def:>9.1f}%")
print(f"  {'Aggregated, balanced':<40}  {acc_agg_bal:>9.1f}%")
print()
print(f"  Aggregated (balanced) PD Recall : {rec_agg_bal:.1f}%")
print(f"  Aggregated (balanced) F1 Score  : {f1_agg_bal:.1f}%")
print()

delta = acc_agg_bal - acc_b
if delta > 2:
    print(f"  BETTER: Aggregated embeddings improve accuracy by +{delta:.1f}pp over per-segment.")
    print("  Concatenating early+middle+late adds discriminative information.")
    print("  Consider using aggregated embeddings as input to Gating Network.")
elif delta < -2:
    print(f"  WORSE: Aggregated embeddings reduce accuracy by {abs(delta):.1f}pp.")
    print("  Concatenation adds noise. Mean-pool or keep per-segment approach.")
else:
    print(f"  SIMILAR: Aggregated vs per-segment within {abs(delta):.1f}pp.")
    print("  Temporal segmentation is still useful — Gating Network can learn weights.")

Running Section 5B: Aggregated embeddings test...

Aggregated metadata: 1247 rows  |  strategy=concat  |  dim=2304
  PD=338  HC=909



Loaded: 1247  missing/corrupt: 0  dim=2304



  AGGREGATED (concat 3x768=2304-dim) vs PER-SEGMENT (768-dim):
  ----------------------------------------------------------
  Method                                      Accuracy
  ----------------------------------------------------------
  Per-segment, default (Section 1B baseline)       73.4%
  Per-segment, balanced (Section 5A)             68.7%
  Aggregated, default                            71.2%
  Aggregated, balanced                           70.3%

  Aggregated (balanced) PD Recall : 45.2%
  Aggregated (balanced) F1 Score  : 45.2%

  SIMILAR: Aggregated vs per-segment within 1.7pp.
  Temporal segmentation is still useful — Gating Network can learn weights.


In [15]:
import torch
import soundfile as sf
import scipy.signal as sp_signal
from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

print("Running Section 5C: Layer analysis — last layer vs avg of last 4 layers...")
print()

DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
SEG_ROOT  = os.path.join(os.path.dirname(BASE), "segments")
TARGET_SR = 16000
N_SAMPLE  = 20   # per group (6 groups = 120 files max)

print(f"  Device     : {DEVICE}")
print(f"  Segments @ : {SEG_ROOT}")
print(f"  Sample     : {N_SAMPLE} files per group")
print()

# ── Load model ────────────────────────────────────────────────────────────
print("  Loading facebook/wav2vec2-base ...")
feat_ex  = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-base")
model_hs = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
model_hs = model_hs.to(DEVICE).eval()
print("  Model loaded.")
print()

# ── Stratified sample — explicit loop, avoids pandas groupby column drops ─
sample_parts = []
for _cls in ["PD", "HC"]:
    for _seg in ["early", "middle", "late"]:
        grp = meta[(meta["class"] == _cls) & (meta["segment"] == _seg)]
        sample_parts.append(grp.sample(min(N_SAMPLE, len(grp)), random_state=42))
sample_df = pd.concat(sample_parts, ignore_index=True)
print(f"  Sampled {len(sample_df)} files ({N_SAMPLE} per group x 6 groups)")
print()

# ── Audio helpers ─────────────────────────────────────────────────────────
def load_and_resample(fpath, target_sr=16000):
    wav, sr = sf.read(fpath)
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != target_sr:
        n_out = int(len(wav) * target_sr / sr)
        wav   = sp_signal.resample(wav, n_out)
    return wav.astype(np.float32)

def extract_last_and_last4(wav, model, extractor, device):
    """
    Returns:
      emb_last  : [768]  mean-pool of last transformer layer
      emb_last4 : [768]  mean-pool of avg(last 4 transformer layers)
    """
    inp = extractor(wav, sampling_rate=TARGET_SR,
                    return_tensors="pt", padding=True)
    with torch.no_grad():
        out = model(inp.input_values.to(device),
                    output_hidden_states=True)
    # hidden_states: tuple of 13 tensors (1, T, 768)
    # idx 0 = CNN feature extractor; 1..12 = transformer layers
    hs       = out.hidden_states
    last_np  = hs[-1][0].cpu().numpy()           # (T, 768) — layer 12
    last4_np = np.mean(
        np.stack([hs[-(i+1)][0].cpu().numpy() for i in range(4)], axis=0),
        axis=0
    )                                             # (T, 768) — avg layers 9-12
    return last_np.mean(axis=0), last4_np.mean(axis=0)

# ── Extraction loop — use iterrows() so row["class"] works as string key ──
embs_last, embs_last4, lyr_labels = [], [], []
skipped = 0

for i, (_, row) in enumerate(sample_df.iterrows(), 1):
    audio_path = os.path.join(SEG_ROOT, row["class"], row["segment"],
                               row["filename"])
    if not os.path.isfile(audio_path):
        skipped += 1
        continue
    try:
        wav = load_and_resample(audio_path)
        e_last, e_last4 = extract_last_and_last4(wav, model_hs, feat_ex, DEVICE)
        embs_last.append(e_last)
        embs_last4.append(e_last4)
        lyr_labels.append(row["class"])
    except Exception:
        skipped += 1
    if i % 30 == 0:
        print(f"  Processed {i}/{len(sample_df)}  skipped={skipped}")

print(f"  Extraction done.  Valid={len(embs_last)}  skipped={skipped}")
print()

if len(embs_last) < 10:
    print("  WARNING: Too few valid files for layer comparison. Check SEG_ROOT path.")
else:
    X_last  = np.stack(embs_last)
    X_last4 = np.stack(embs_last4)
    y_lyr   = (np.array(lyr_labels) == "PD").astype(int)

    print(f"  PD={y_lyr.sum()}  HC={(1-y_lyr).sum()}  dim={X_last.shape[1]}")
    print()

    cv_lyr = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    pl = Pipeline([("sc", StandardScaler()),
                   ("clf", LogisticRegression(max_iter=1000, random_state=42,
                                              class_weight="balanced"))])

    acc_last  = cross_val_score(pl, X_last,  y_lyr, cv=cv_lyr, scoring="accuracy").mean() * 100
    acc_last4 = cross_val_score(pl, X_last4, y_lyr, cv=cv_lyr, scoring="accuracy").mean() * 100
    rec_last  = cross_val_score(pl, X_last,  y_lyr, cv=cv_lyr, scoring="recall").mean()   * 100
    rec_last4 = cross_val_score(pl, X_last4, y_lyr, cv=cv_lyr, scoring="recall").mean()   * 100
    f1_last   = cross_val_score(pl, X_last,  y_lyr, cv=cv_lyr, scoring="f1").mean()       * 100
    f1_last4  = cross_val_score(pl, X_last4, y_lyr, cv=cv_lyr, scoring="f1").mean()       * 100

    print(f"  LAYER ANALYSIS RESULTS  (n={len(embs_last)}, class_weight=balanced)")
    print("  " + "-" * 62)
    print(f"  {'Strategy':<35}  {'Accuracy':>10}  {'PD Recall':>10}  {'F1':>8}")
    print("  " + "-" * 62)
    print(f"  {'Last layer only  (current pipeline)':<35}  {acc_last:>9.1f}%  {rec_last:>9.1f}%  {f1_last:>7.1f}%")
    print(f"  {'Avg of last 4 layers':<35}  {acc_last4:>9.1f}%  {rec_last4:>9.1f}%  {f1_last4:>7.1f}%")
    print()

    d_acc = acc_last4 - acc_last
    d_rec = rec_last4 - rec_last

    if d_rec > 3 or d_acc > 3:
        print(f"  RECOMMENDATION: Switch to last-4-layer average.")
        print(f"  Gain -> Accuracy {d_acc:+.1f}pp  |  PD Recall {d_rec:+.1f}pp")
        print("  Re-extract all embeddings with output_hidden_states=True,")
        print("  average layers [-4:] before mean-pooling over time.")
    elif d_rec < -3 or d_acc < -3:
        print(f"  RECOMMENDATION: Keep last layer only — last-4-avg is WORSE.")
        print(f"  Delta -> Accuracy {d_acc:+.1f}pp  |  PD Recall {d_rec:+.1f}pp")
    else:
        print(f"  RECOMMENDATION: No significant difference (within 3pp).")
        print("  Current last-layer mean-pool strategy is sufficient.")
        print("  Layer averaging unlikely to justify re-extraction overhead.")

Running Section 5C: Layer analysis — last layer vs avg of last 4 layers...

  Device     : cpu
  Segments @ : /home/bs00956/Desktop/Personal/PD/Pipeline-Implementation/segments
  Sample     : 20 files per group

  Loading facebook/wav2vec2-base ...


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Model loaded.

  Sampled 120 files (20 per group x 6 groups)



  Processed 30/120  skipped=0


  Processed 60/120  skipped=0


  Processed 90/120  skipped=0


  Processed 120/120  skipped=0
  Extraction done.  Valid=120  skipped=0

  PD=60  HC=60  dim=768



  LAYER ANALYSIS RESULTS  (n=120, class_weight=balanced)
  --------------------------------------------------------------
  Strategy                               Accuracy   PD Recall        F1
  --------------------------------------------------------------
  Last layer only  (current pipeline)       70.8%       73.3%     71.5%
  Avg of last 4 layers                      72.5%       75.0%     72.9%

  RECOMMENDATION: No significant difference (within 3pp).
  Current last-layer mean-pool strategy is sufficient.
  Layer averaging unlikely to justify re-extraction overhead.
